# Notebook 03 — Evaluate All Experiments
**Member: C** | Set `model.checkpoint_path` in `configs/eval.yaml` before each run.

| Experiment | `checkpoint_path` | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                             | GSM8K |
| Exp 1 few-shot CoT | `null`                             | GSM8K |
| Exp 2 adaptation   | `results/checkpoints/adaptation`   | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation` | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`     | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered`     | ARC |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
token = 'ghp_xxxxxxxxxxxx'  # paste your GitHub token each session
repo_url = f'https://{token}@github.com/TIFFMUN/SC4001-Group133.git'

if not os.path.exists('/content/SC4001-Group133/.git'):
    os.system(f'git clone {repo_url} /content/SC4001-Group133')

os.chdir('/content/SC4001-Group133')
os.system(f'git pull {repo_url} main')
!pip install -r requirements.txt -q
!pip install -e . -q  # installs src/ as a package, fixes import issues on Colab Python 3.13

In [ ]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

In [ ]:
# Exp 0 — zero-shot baseline
from src.baselines.cot_prompting import run_baseline
from src.evaluation.metrics import compute_summary_stats, save_results

results = run_baseline(mode='zero_shot', dataset_name='gsm8k')
save_results(results, 'results/exp0_zero_shot_gsm8k.csv')
print(compute_summary_stats(results))

In [ ]:
# Exp 1 — few-shot CoT baseline
results = run_baseline(mode='few_shot', dataset_name='gsm8k')
save_results(results, 'results/exp1_few_shot_gsm8k.csv')
print(compute_summary_stats(results))

In [ ]:
# Exp 2-4 — set checkpoint_path in configs/eval.yaml, update EXPERIMENT_NAME, then run
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # change per run

from src.evaluation.eval_gsm8k import evaluate_gsm8k
evaluate_gsm8k(experiment_name=EXPERIMENT_NAME)

In [ ]:
# Exp 5 — ARC cross-task transfer (uses Exp 4 checkpoint)
from src.evaluation.eval_arc import evaluate_arc
evaluate_arc(experiment_name='exp5_d3_cross_task')

In [ ]:
# List all saved CSVs on Drive
csvs = [f for f in os.listdir('/content/drive/MyDrive/SC4001/results') if f.endswith('.csv')]
for f in sorted(csvs):
    print(f)

## Exp 6 — Inference-gated budget router (optional, run after Exps 0–5)

Prerequisite: set `router.enabled: true` and `router.checkpoint_path` in `configs/eval.yaml`.

The router sweeps a range of complexity thresholds and saves a Pareto-ready CSV to
`results/exp6_router_sweep.csv`. Use `max_samples=50` for a quick sanity check first.

In [ ]:
# Exp 6 — budget router threshold sweep
# Step 1: quick sanity check with 50 samples
from src.evaluation.eval_router import run_router_sweep

sweep_df = run_router_sweep(experiment_name='exp6_router', max_samples=50)
print(sweep_df[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))

In [ ]:
# Exp 6 — full sweep (set max_samples: null in eval.yaml for full 1319-sample eval)
# After confirming the sanity check above looks reasonable, run:
sweep_df_full = run_router_sweep(experiment_name='exp6_router_full', max_samples=None)
print(sweep_df_full[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))